[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/graph_theory/05_flows_matchings_and_bipartite_graphs/first_principles.ipynb)

# Topic 05: Flows, Matchings and Bipartite Graphs

## 1. First-Principles Intuition & Motivation

A graph so far has been a map of *what is connected to what*. Now attach a number $c(e)$ to every arc — the maximum rate at which that link can carry something — and ask a physical question: **how much can flow from a source $s$ to a sink $t$?**

Two intuitions collide productively.

- **From the supply side**: push material along $s$–$t$ routes until every route is blocked by some saturated arc. This is constructive and gives *lower* bounds on the answer.
- **From the obstruction side**: pick a set of arcs whose removal separates $s$ from $t$. Nothing can flow past it, so its total capacity is an *upper* bound on the answer.

The max-flow min-cut theorem says these two bounds always meet. Any maximum flow is certified optimal by a cut of exactly the same value, and any minimum cut is certified by a flow. That is remarkable: a *continuous* optimization (choose real numbers $f(e)$) is solved exactly by a *combinatorial* object (choose a set $S$ of vertices), with no gap. It is linear-programming duality, discovered combinatorially a year before the general theory was digested.

### Why naive greed fails, and what fixes it

Consider four vertices $s, a, b, t$ with unit capacities on $s \to a$, $s \to b$, $a \to b$, $a \to t$, $b \to t$. Greedily route one unit along $s \to a \to b \to t$. Both $s \to a$ and $b \to t$ are now saturated, so no further $s$–$t$ route survives in the *original* graph — yet the true maximum is $2$, achieved by $s \to a \to t$ and $s \to b \to t$.

What went wrong is that the first decision was irrevocable. The repair is the **residual graph**: alongside each arc with remaining capacity $c(e) - f(e)$, install a *reverse* arc with capacity $f(e)$, meaning "you may cancel up to $f(e)$ units of what you already sent." In the example the residual reverse arc $b \to a$ lets us find the augmenting path $s \to b \to a \to t$, whose effect is to *reroute*, not merely to add.

> Reverse arcs are not pipes. They are the algorithmic representation of regret.

With residual arcs, "no $s$–$t$ path remains" upgrades from a heuristic stopping rule to a genuine optimality certificate — that is the content of Theorem C below.

### From flows to matchings

Now a seemingly unrelated problem: $n$ workers, $n$ jobs, and a bipartite graph of who can do what. Pick as many disjoint worker–job pairs as possible. This is **maximum bipartite matching**.

Model it as a flow: add a source $s$ with unit-capacity arcs to every worker, orient every compatibility edge from workers to jobs with capacity $1$, and add unit-capacity arcs from every job to a sink $t$. An integral flow of value $k$ is exactly a matching of size $k$, because unit capacities force each worker and each job to carry at most one unit. The **integrality theorem** guarantees an integral maximum flow exists, so max-flow algorithms solve matching directly.

Everything then specializes beautifully:

- augmenting paths in the residual graph become **alternating paths** that flip matched and unmatched edges (Berge's theorem);
- minimum cuts become **minimum vertex covers** (König's theorem);
- the condition for saturating one side becomes **Hall's marriage condition**;
- adding costs to edges turns the problem into the **assignment problem**, whose dual variables are potentials and whose algorithm — the Hungarian method — is the combinatorial ancestor of discrete optimal transport.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition (flow network).** A **flow network** is a tuple $(G, c, s, t)$ where $G = (V,E)$ is a digraph, $c : E \to \mathbb{R}_{\ge 0}$ is a capacity function, and $s \neq t$ are the source and sink. Write $n = \vert V \vert$, $m = \vert E \vert$.

**Definition (flow).** A function $f : E \to \mathbb{R}$ is a **flow** if

$$
0 \le f(e) \le c(e) \quad \forall e \in E \qquad \text{(capacity)}
$$

$$
\sum_{e \in \delta^{-}(v)} f(e) = \sum_{e \in \delta^{+}(v)} f(e) \quad \forall v \in V \setminus \lbrace s,t \rbrace \qquad \text{(conservation)}
$$

Its **value** is $\vert f \vert = \sum_{e \in \delta^{+}(s)} f(e) - \sum_{e \in \delta^{-}(s)} f(e)$, the net outflow of the source.

**Definition (cut).** An **$s$–$t$ cut** is a partition $V = S \cup T$ with $s \in S$, $t \in T$. Its **capacity** is $c(S,T) = \sum_{u \in S,\, v \in T,\, (u,v) \in E} c(u,v)$ — only forward-crossing arcs count.

**Definition (residual network).** Given a flow $f$, the residual network $G_f$ has, for each arc $(u,v) \in E$, a forward arc $(u,v)$ of residual capacity $c_f(u,v) = c(u,v) - f(u,v)$ when positive, and a backward arc $(v,u)$ of residual capacity $c_f(v,u) = f(u,v)$ when positive. An **augmenting path** is a directed $s$–$t$ path in $G_f$; its **bottleneck** is the minimum residual capacity along it.

**Definition (matching, cover, saturation).** A **matching** $M \subseteq E$ is a set of pairwise non-adjacent edges; $\nu(G) = \max \vert M \vert$. A **vertex cover** $C \subseteq V$ meets every edge; $\tau(G) = \min \vert C \vert$. A matching **saturates** $X \subseteq V$ if every vertex of $X$ is an endpoint of some edge of $M$; it is **perfect** if it saturates $V$. An **$M$-alternating path** alternates between edges outside and inside $M$; it is **$M$-augmenting** if both endpoints are unmatched.

**Definition (assignment problem).** Given a cost matrix $c \in \mathbb{R}^{n \times n}$, find a permutation $\sigma$ minimizing $\sum_{i=1}^{n} c_{i \sigma(i)}$; equivalently minimize $\langle c, X \rangle$ over the Birkhoff polytope of doubly stochastic $X$.

### Theorem statements

**Theorem A (flow-value lemma).** For every flow $f$ and every $s$–$t$ cut $(S,T)$,

$$
\vert f \vert = f(S,T) - f(T,S) \le c(S,T)
$$

where $f(S,T)$ sums $f$ over arcs from $S$ to $T$. (Weak duality: every flow value is below every cut capacity.)

**Theorem B (augmentation).** If $P$ is an augmenting path with bottleneck $\beta \gt 0$, then pushing $\beta$ along $P$ (adding on forward arcs, subtracting on backward arcs) yields a flow of value $\vert f \vert + \beta$.

**Theorem C (max-flow min-cut, Ford–Fulkerson 1956).** For a flow $f$ the following are equivalent: (i) $f$ is a maximum flow; (ii) $G_f$ contains no augmenting path; (iii) $\vert f \vert = c(S,T)$ for some $s$–$t$ cut. Consequently

$$
\max_{f} \vert f \vert = \min_{(S,T)} c(S,T)
$$

**Theorem D (integrality).** If all capacities are integers, some maximum flow is integral, and Ford–Fulkerson computes one in at most $\vert f^{\ast} \vert$ augmentations.

**Theorem E (Edmonds–Karp).** If every augmenting path is chosen to have the fewest arcs (BFS in $G_f$), the number of augmentations is $O(nm)$ and the total running time is $O(nm^2)$, independent of the capacity values.

**Theorem F (Berge, 1957).** A matching $M$ is maximum if and only if $G$ contains no $M$-augmenting path.

**Theorem G (König, 1931).** In a bipartite graph, $\nu(G) = \tau(G)$: the maximum matching size equals the minimum vertex cover size.

**Theorem H (Hall, 1935).** A bipartite graph with parts $X$ and $Y$ has a matching saturating $X$ if and only if $\vert N(S) \vert \ge \vert S \vert$ for every $S \subseteq X$.

**Theorem I (Birkhoff–von Neumann).** The extreme points of the polytope of doubly stochastic matrices are exactly the permutation matrices; hence the assignment LP has an integral optimum and the Hungarian algorithm solves it in $O(n^3)$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1: The flow-value lemma (weak duality)

**Theorem A.** For every flow $f$ and every $s$–$t$ cut $(S,T)$: $\vert f \vert = f(S,T) - f(T,S) \le c(S,T)$.

**Proof.** Define the **excess** $\mathrm{ex}(v) = \sum_{e \in \delta^{+}(v)} f(e) - \sum_{e \in \delta^{-}(v)} f(e)$ (outflow minus inflow). Conservation says $\mathrm{ex}(v) = 0$ for $v \notin \lbrace s,t \rbrace$, and $\mathrm{ex}(s) = \vert f \vert$ by definition. Summing over $S$:

$$
\sum_{v \in S} \mathrm{ex}(v) = \mathrm{ex}(s) = \vert f \vert
$$

because every other vertex of $S$ contributes zero. Now expand the left side arc by arc. An arc with **both** endpoints in $S$ appears once with a $+$ (at its tail) and once with a $-$ (at its head) and cancels. An arc from $S$ to $T$ contributes $+f(e)$; an arc from $T$ to $S$ contributes $-f(e)$. Hence

$$
\vert f \vert = f(S,T) - f(T,S)
$$

Finally $f(S,T) \le c(S,T)$ by capacity and $f(T,S) \ge 0$ by nonnegativity, so $\vert f \vert \le c(S,T)$. $\blacksquare$

$$
\boxed{\vert f \vert \le c(S,T) \quad \text{for every flow } f \text{ and every cut } (S,T)}
$$

Two consequences used constantly: the value can be measured across *any* cut, not just at $s$; and any flow/cut pair with equal value certifies **both** optimality claims simultaneously.

### Proof 2: Max-flow min-cut

**Theorem C.** (i) $f$ maximum $\iff$ (ii) no augmenting path in $G_f$ $\iff$ (iii) $\vert f \vert = c(S,T)$ for some cut.

**Proof.**

**(i) $\Rightarrow$ (ii).** Contrapositive. If $G_f$ has an $s$–$t$ path $P$, its bottleneck $\beta = \min_{e \in P} c_f(e)$ is positive (all residual capacities on $P$ are $\gt 0$ by construction). Pushing $\beta$ along $P$ preserves capacities (forward arcs gain at most their residual, backward arcs lose at most their current flow) and preserves conservation (each interior vertex of $P$ gains $\beta$ on one incident arc and loses $\beta$ on another, in all four forward/backward combinations). The value rises to $\vert f \vert + \beta$, so $f$ was not maximum. This also proves **Theorem B**.

**(ii) $\Rightarrow$ (iii).** Let $S = \lbrace v : v \text{ reachable from } s \text{ in } G_f \rbrace$ and $T = V \setminus S$. By hypothesis $t \notin S$, so $(S,T)$ is a genuine $s$–$t$ cut. Take any arc $(u,v) \in E$ with $u \in S$, $v \in T$. If $f(u,v) \lt c(u,v)$ the residual forward arc $(u,v)$ would exist and $v$ would be reachable — contradiction. Hence

$$
f(u,v) = c(u,v) \quad \text{for every arc from } S \text{ to } T
$$

Similarly, for an arc $(v,u) \in E$ with $v \in T$, $u \in S$: if $f(v,u) \gt 0$ the residual backward arc $(u,v)$ would exist and $v$ would be reachable — contradiction. Hence $f(v,u) = 0$ for every arc from $T$ to $S$. Substituting into Theorem A:

$$
\vert f \vert = f(S,T) - f(T,S) = c(S,T) - 0 = c(S,T)
$$

**(iii) $\Rightarrow$ (i).** By weak duality every flow $g$ satisfies $\vert g \vert \le c(S,T) = \vert f \vert$, so $f$ is maximum (and $(S,T)$ is a minimum cut). $\blacksquare$

$$
\boxed{\max_{f} \vert f \vert = \min_{(S,T)} c(S,T)}
$$

**How to read off the min cut in practice:** run any max-flow algorithm, then do one graph search from $s$ in the final residual network. The reachable set *is* $S$. This costs $O(m)$ and is the standard way to obtain segmentations, bottleneck diagnoses, and vertex covers.

### Proof 3: Integrality, and why combinatorics inherits flow theory

**Theorem D.** With integer capacities, Ford–Fulkerson maintains an integral flow at every step and terminates with an integral maximum flow.

**Proof (induction on augmentations).** The zero flow is integral. If $f$ is integral, all residual capacities $c(u,v) - f(u,v)$ and $f(u,v)$ are integers, so the bottleneck $\beta$ of any augmenting path is a positive **integer**; pushing $\beta$ keeps every arc value integral. Each augmentation increases $\vert f \vert$ by $\beta \ge 1$, and $\vert f \vert$ is bounded above by $c(\lbrace s \rbrace, V \setminus \lbrace s \rbrace)$, so the process halts after at most $\vert f^{\ast} \vert$ augmentations, at which point no augmenting path exists and Theorem C makes the flow maximum. $\blacksquare$

$$
\boxed{\text{integer capacities} \Rightarrow \text{some maximum flow is integral}}
$$

**Why this matters.** The flow LP is *not* obviously integral — but its constraint matrix is the incidence matrix of a digraph, which is **totally unimodular**, so every vertex of the feasible polytope has integer coordinates when capacities are integers. Consequently a whole family of selection problems can be solved by flow with no rounding loss:

| Combinatorial problem | Flow model |
|---|---|
| Maximum bipartite matching | unit capacities, $s \to X \to Y \to t$ |
| Maximum arc-disjoint $s$–$t$ paths | all capacities $1$ (Menger's theorem) |
| Maximum vertex-disjoint paths | split each $v$ into $v_{\text{in}} \to v_{\text{out}}$ with capacity $1$ |
| Bipartite $b$-matching / scheduling | capacities $b(v)$ on the source and sink arcs |
| Project selection / max closure | infinite-capacity precedence arcs, min cut selects the project set |

**Caution.** Integrality is about the *existence* of an integral optimum, not about speed: Ford–Fulkerson with poor path choices is only *pseudo-polynomial*, taking $\Theta(\vert f^{\ast} \vert)$ augmentations on the classic "two big arcs joined by a unit arc" network. With irrational capacities it may not terminate at all.

### Proof 4: The Edmonds–Karp bound $O(nm^2)$

**Theorem E.** If each augmentation uses a shortest (fewest-arc) augmenting path found by BFS in $G_f$, then at most $O(nm)$ augmentations occur, so the algorithm runs in $O(nm^2)$.

Let $\delta_f(s,v)$ denote the number of arcs on a shortest $s$–$v$ path in the residual network $G_f$ ($+\infty$ if unreachable).

**Lemma 1 (residual distances never decrease).** If $f'$ is obtained from $f$ by one shortest-path augmentation, then $\delta_{f'}(s,v) \ge \delta_f(s,v)$ for all $v$.

*Proof.* Suppose not, and among all violating vertices choose $v$ with $\delta_{f'}(s,v)$ minimum. Let $u$ be the predecessor of $v$ on a shortest $s$–$v$ path in $G_{f'}$, so $\delta_{f'}(s,v) = \delta_{f'}(s,u) + 1$ and, by minimality of $v$, $\delta_{f'}(s,u) \ge \delta_f(s,u)$.

*Case 1: $(u,v) \in G_f$.* Then $\delta_f(s,v) \le \delta_f(s,u) + 1 \le \delta_{f'}(s,u) + 1 = \delta_{f'}(s,v)$, contradicting the choice of $v$.

*Case 2: $(u,v) \notin G_f$ but $(u,v) \in G_{f'}$.* The arc appeared because the augmentation pushed flow along the reverse arc $(v,u)$, which therefore lay on the shortest augmenting path in $G_f$: $\delta_f(s,u) = \delta_f(s,v) + 1$. Then

$$
\delta_f(s,v) = \delta_f(s,u) - 1 \le \delta_{f'}(s,u) - 1 = \delta_{f'}(s,v) - 2
$$

again contradicting $\delta_{f'}(s,v) \lt \delta_f(s,v)$. $\square$

**Lemma 2 (each arc is critical $O(n)$ times).** Call an arc $(u,v)$ **critical** for an augmentation if it is the bottleneck, hence disappears from the residual network. When $(u,v)$ is critical it lies on a shortest augmenting path, so $\delta_f(s,v) = \delta_f(s,u) + 1$. For $(u,v)$ to become critical again it must first reappear, which requires a later augmentation to push flow along $(v,u)$; at that moment $\delta_{f'}(s,u) = \delta_{f'}(s,v) + 1$. Using Lemma 1 twice,

$$
\delta_{f'}(s,u) = \delta_{f'}(s,v) + 1 \ge \delta_f(s,v) + 1 = \delta_f(s,u) + 2
$$

So $\delta(s,u)$ increases by at least $2$ between consecutive criticalities of $(u,v)$. Since distances are integers in $\lbrace 0, 1, \dots, n-1 \rbrace$ while $u$ remains reachable, each arc can be critical at most $n/2$ times. $\square$

**Conclusion.** Every augmentation makes at least one arc critical, and $G_f$ contains at most $2m$ arcs, so the number of augmentations is at most $2m \cdot n/2 = O(nm)$. Each augmentation costs one BFS, $O(m)$:

$$
\boxed{T_{\text{Edmonds–Karp}} = O(n m^2), \text{ independent of the capacity magnitudes}}
$$

**Dinic's refinement.** Group augmentations into *phases* of equal residual distance and saturate a whole **blocking flow** per phase. Distances strictly increase across phases, so there are at most $n$ phases; a blocking flow costs $O(nm)$, giving $O(n^2 m)$. On unit-capacity networks — bipartite matching — the phase count drops to $O(\sqrt{n})$, yielding the Hopcroft–Karp bound $O(m \sqrt{n})$.

### Proof 5: Berge's augmenting-path theorem

**Theorem F.** A matching $M$ is maximum if and only if there is no $M$-augmenting path.

**Proof.**

($\Rightarrow$) If $P$ is $M$-augmenting, it has odd length $2k+1$, starts and ends at unmatched vertices, and alternates unmatched/matched edges, containing $k+1$ edges outside $M$ and $k$ inside. The symmetric difference $M \triangle E(P)$ is again a matching (the only vertices whose incident matched edge changed are interior to $P$, each still covered exactly once, and the two endpoints become matched) of size $\vert M \vert + 1$. So $M$ was not maximum.

($\Leftarrow$) Suppose $M$ is not maximum and let $M'$ be a larger matching. Consider $H = M \triangle M'$. Every vertex has degree at most $2$ in $H$ (at most one edge from each matching), so $H$ is a disjoint union of paths and cycles, and along each of them the edges alternate between $M$ and $M'$. Alternating cycles have even length and contain equally many edges of each matching. Since $\vert M' \vert \gt \vert M \vert$, some component must contain more $M'$-edges than $M$-edges; it can only be a path, and it must begin and end with $M'$-edges — that is, an **$M$-augmenting path**. $\blacksquare$

$$
\boxed{M \text{ maximum} \iff \text{no } M\text{-augmenting path exists}}
$$

**Algorithmic reading.** In bipartite graphs, alternating paths are exactly residual $s$–$t$ paths in the unit-capacity flow model, so BFS/DFS finds them in $O(m)$ and repeated augmentation gives $O(nm)$; Hopcroft–Karp batches vertex-disjoint shortest augmenting paths to reach $O(m\sqrt{n})$. In *general* graphs, odd cycles create "blossoms" that must be contracted — Edmonds' 1965 blossom algorithm, the paper that introduced the notion of a polynomial-time algorithm.

### Proof 6: König's theorem via minimum cuts

**Theorem G.** For bipartite $G$ with parts $X$ and $Y$: $\nu(G) = \tau(G)$.

**Proof.**

*Weak direction ($\nu \le \tau$).* Every edge of a matching must be covered, and distinct matching edges share no vertex, so a cover needs at least one distinct vertex per matching edge.

*Construction.* Build the flow network $N$: arcs $s \to x$ of capacity $1$ for $x \in X$; arcs $x \to y$ of capacity $+\infty$ for every edge $xy \in E$; arcs $y \to t$ of capacity $1$ for $y \in Y$. By integrality (Theorem D) a maximum integral flow uses each arc $0$ or $1$ times, and the saturated middle arcs form a matching with $\vert M \vert = \vert f^{\ast} \vert$; conversely any matching yields a flow of its size. Hence

$$
\nu(G) = \vert f^{\ast} \vert
$$

*From a min cut to a cover.* Let $(S,T)$ be a minimum cut, necessarily of finite capacity, and set

$$
C = (X \cap T) \cup (Y \cap S)
$$

**$C$ is a vertex cover.** Take any edge $xy$ with $x \in X$, $y \in Y$. If $x \notin C$ then $x \in S$, and if $y \notin C$ then $y \in T$; but then the arc $x \to y$ crosses the cut with capacity $+\infty$, contradicting finiteness. So every edge meets $C$.

**Its size is the cut capacity.** The only finite-capacity arcs crossing from $S$ to $T$ are $s \to x$ with $x \in T$ and $y \to t$ with $y \in S$, each of capacity $1$:

$$
c(S,T) = \vert X \cap T \vert + \vert Y \cap S \vert = \vert C \vert
$$

Therefore $\tau(G) \le \vert C \vert = c(S,T) = \vert f^{\ast} \vert = \nu(G)$ by max-flow min-cut, and combined with $\nu \le \tau$:

$$
\boxed{\nu(G) = \tau(G) \quad \text{for bipartite } G}
$$

$\blacksquare$

**Sharpness.** The triangle $K_3$ has $\nu = 1$, $\tau = 2$: König genuinely needs bipartiteness. The obstruction is exactly the odd cycle, which is also what breaks integrality of the matching LP and forces Edmonds' blossoms in the general case. The general replacement is the Tutte–Berge formula $\nu(G) = \tfrac{1}{2}\big( n - \max_{U \subseteq V} ( \mathrm{odd}(G - U) - \vert U \vert ) \big)$.

### Proof 7: Hall's marriage theorem

**Theorem H.** A bipartite graph with parts $X, Y$ has a matching saturating $X$ iff $\vert N(S) \vert \ge \vert S \vert$ for all $S \subseteq X$, where $N(S)$ is the set of neighbours of $S$.

**Proof.**

($\Rightarrow$) If $M$ saturates $X$, the map sending $x \in S$ to its partner $M(x)$ is injective and lands in $N(S)$, so $\vert N(S) \vert \ge \vert S \vert$.

($\Leftarrow$) Contrapositive: assume no matching saturates $X$, i.e. $\nu(G) \lt \vert X \vert$. By König (Theorem G) there is a vertex cover $C$ with $\vert C \vert = \nu(G) \lt \vert X \vert$. Split it as $C = C_X \cup C_Y$ with $C_X = C \cap X$, $C_Y = C \cap Y$, and put

$$
S = X \setminus C_X
$$

Every edge leaving $S$ must be covered by $C$; its $X$-endpoint is not in $C$, so its $Y$-endpoint lies in $C_Y$. Hence $N(S) \subseteq C_Y$ and

$$
\vert N(S) \vert \le \vert C_Y \vert = \vert C \vert - \vert C_X \vert \lt \vert X \vert - \vert C_X \vert = \vert S \vert
$$

so Hall's condition fails for this $S$. $\blacksquare$

$$
\boxed{\text{matching saturating } X \iff \vert N(S) \vert \ge \vert S \vert \ \ \forall S \subseteq X}
$$

**Deficiency version.** The same argument gives $\nu(G) = \vert X \vert - \max_{S \subseteq X} \big( \vert S \vert - \vert N(S) \vert \big)$: the maximum matching falls short of saturating $X$ by exactly the worst *deficiency*. This is the certificate returned by matching codes when they fail — a witness set $S$ whose neighbourhood is too small.

**Corollary (regular bipartite graphs).** Every $k$-regular bipartite graph with $k \ge 1$ has a perfect matching: counting edges between $S$ and $N(S)$ gives $k\vert S \vert \le k \vert N(S) \vert$. Iterating, its edge set decomposes into $k$ perfect matchings — the combinatorial core of Birkhoff's theorem and of round-robin scheduling.

### Proof 8: The assignment problem, LP duality and the Hungarian algorithm

**Primal (assignment LP).** Minimize $\sum_{i,j} c_{ij} x_{ij}$ subject to $\sum_j x_{ij} = 1$, $\sum_i x_{ij} = 1$, $x_{ij} \ge 0$ — i.e. minimize $\langle c, X \rangle$ over the **Birkhoff polytope** of doubly stochastic matrices.

**Theorem I (Birkhoff–von Neumann).** The extreme points of that polytope are the permutation matrices.

*Proof sketch.* A doubly stochastic $X$ that is not a permutation has a fractional entry; following fractional entries through rows and columns (each fractional entry forces another in its row and another in its column) produces a cycle of even length in the bipartite support graph. Adding $\pm \varepsilon$ alternately around that cycle keeps all row and column sums equal to $1$ and stays feasible for small $\varepsilon$, exhibiting $X$ as the midpoint of two distinct feasible points — so $X$ is not extreme. Conversely permutation matrices are $0/1$ vertices of the unit cube intersected with the constraints, hence extreme. $\square$

Since a linear objective attains its minimum at an extreme point, the LP optimum is a permutation: **relaxation is exact**.

**Dual.** Maximize $\sum_i u_i + \sum_j v_j$ subject to

$$
u_i + v_j \le c_{ij} \quad \forall i,j
$$

Weak duality is immediate: for any feasible $X$ and any feasible $(u,v)$,

$$
\langle c, X \rangle \ge \sum_{i,j} (u_i + v_j) x_{ij} = \sum_i u_i \sum_j x_{ij} + \sum_j v_j \sum_i x_{ij} = \sum_i u_i + \sum_j v_j
$$

**Complementary slackness.** A pair $(X, u, v)$ is optimal iff $x_{ij} \gt 0$ implies $u_i + v_j = c_{ij}$ — the assignment uses only **tight** edges.

**The Hungarian algorithm** turns this into a procedure: maintain dual-feasible potentials $(u,v)$ and a matching $M$ inside the *equality subgraph* $G_{=} = \lbrace (i,j) : u_i + v_j = c_{ij} \rbrace$. Repeat:

1. Search $G_{=}$ for an $M$-augmenting path from an unmatched row (Hungarian trees / alternating BFS). If found, augment — $\vert M \vert$ grows by one.
2. If not, let $R$ be the reached rows and $Q$ the reached columns, and compute the minimum slack escaping the tree,
$$
\theta = \min \lbrace c_{ij} - u_i - v_j : i \in R,\ j \notin Q \rbrace \gt 0
$$
Update $u_i \leftarrow u_i + \theta$ for $i \in R$ and $v_j \leftarrow v_j - \theta$ for $j \in Q$. Dual feasibility is preserved, all currently tight tree edges stay tight, the dual objective increases by $\theta \cdot (\vert R \vert - \vert Q \vert) \gt 0$, and at least one new edge becomes tight — so progress is guaranteed.

After $n$ successful augmentations the matching is perfect and complementary slackness certifies optimality. Each phase costs $O(n^2)$:

$$
\boxed{\min_{\sigma} \sum_i c_{i\sigma(i)} = \max_{u_i + v_j \le c_{ij}} \Big( \sum_i u_i + \sum_j v_j \Big), \quad \text{computable in } O(n^3)}
$$

**Recognize the pattern.** The potentials $u_i, v_j$ play exactly the role of the shortest-path potentials in Johnson's reweighting and of the heuristic $h$ in A$^{\ast}$ (Topic 04): a dual variable that makes reduced costs $c_{ij} - u_i - v_j$ nonnegative and lets a greedy primal method proceed safely.

## 4. Computational & Algorithmic Insights

### Algorithm comparison

| Problem | Algorithm | Time | Notes |
|---|---|---|---|
| Max flow | Ford–Fulkerson (arbitrary paths) | $O(m \vert f^{\ast} \vert)$ | pseudo-polynomial; may not terminate on irrational capacities |
| Max flow | Edmonds–Karp (BFS paths) | $O(n m^2)$ | strongly polynomial, capacity-independent |
| Max flow | Dinic (blocking flows) | $O(n^2 m)$ | $O(m \sqrt{n})$ on unit capacities |
| Max flow | Push–relabel with highest label | $O(n^2 \sqrt{m})$ | best classical general-purpose choice |
| Max flow | Orlin (2013) / interior point (2022) | $O(nm)$ / near-linear | theoretical frontier |
| Bipartite matching | augmenting paths | $O(nm)$ | one search per matched pair |
| Bipartite matching | Hopcroft–Karp | $O(m \sqrt{n})$ | phase-batched shortest augmenting paths |
| General matching | Edmonds' blossom | $O(n^3)$ | contracts odd cycles |
| Assignment (min cost) | Hungarian / Kuhn–Munkres | $O(n^3)$ | maintains dual potentials |
| Min-cost flow | successive shortest paths + potentials | $O(\vert f^{\ast} \vert \cdot (m + n \log n))$ | Johnson reweighting inside |
| Entropic optimal transport | Sinkhorn iterations | $O(n^2 / \varepsilon)$ per accuracy $\varepsilon$ | differentiable relaxation of assignment |

### Modeling toolbox

Most "hard-looking" combinatorial constraints are expressible as flow gadgets:

| Requirement | Gadget |
|---|---|
| Vertex capacity $b(v)$ | split $v$ into $v_{\text{in}} \to v_{\text{out}}$ with capacity $b(v)$ |
| Several sources/sinks | super-source and super-sink with infinite arcs |
| Undirected edge of capacity $c$ | two opposite arcs of capacity $c$ each |
| Lower bounds $\ell(e) \le f(e)$ | circulation with demands; solve a feasibility flow first |
| "Must select all prerequisites" | infinite-capacity precedence arcs; min cut = optimal closure |
| Costs as well as capacities | min-cost flow; shortest augmenting path under reduced costs |

### Pitfalls and verification strategy

- **Certify, do not trust.** After computing a claimed maximum flow, verify in $O(m)$: (a) capacity and conservation hold; (b) BFS from $s$ in $G_f$ does not reach $t$; (c) the reachable set $S$ gives $c(S,T) = \vert f \vert$. Steps (b)–(c) are the optimality proof, not a heuristic check.
- **Infinite capacities need care.** Use a finite surrogate larger than $\sum_e c(e)$ so overflow cannot occur, and confirm no min cut uses such an arc.
- **Antiparallel arcs.** Many implementations store arcs in pairs (arc, reverse arc) with an XOR index trick; feeding both $(u,v)$ and $(v,u)$ from the input without splitting one of them silently corrupts residual bookkeeping.
- **Matching from flow.** Recover the matching as the set of middle arcs carrying one unit — never as the set of *saturated* arcs, since source and sink arcs are saturated too.
- **Ties and non-uniqueness.** Both maximum flows and minimum cuts may be non-unique; the min cut given by residual reachability is the *source-side minimal* one, and the sink-side minimal cut comes from reverse reachability from $t$. Reporting which one you used matters in segmentation applications.
- **Numerical capacities.** With floating-point capacities, augmenting can loop on ever-smaller bottlenecks; scale to integers or impose a minimum bottleneck $\varepsilon$.
- **Sanity bounds.** $\nu(G) \le \min(\vert X \vert, \vert Y \vert)$; $\vert f^{\ast} \vert \le \min\big( c(\delta^{+}(s)), c(\delta^{-}(t)) \big)$; Hall deficiency and König cover sizes must agree with the matching size.

## 5. Real-World Physics & AI/ML Applications

### Physics, engineering and operations

- **Transport and utility networks.** Pipeline throughput, power dispatch and packet routing are literal max-flow problems; the min cut names the bottleneck to upgrade, which is why network planners compute the cut, not just the flow value.
- **Reliability and Menger's theorem.** The maximum number of arc-disjoint $s$–$t$ paths equals the minimum number of arcs whose removal disconnects them — unit-capacity max-flow min-cut. This quantifies fault tolerance in data-center topologies and in molecular interaction networks.
- **Percolation and porous media.** Maximum flow through a random capacity field is a classic model of fluid transport in disordered media; the min cut is the "weakest surface" and its scaling connects to first-passage percolation.
- **Statistical mechanics of matching.** For random $n \times n$ costs drawn i.i.d. uniform on $[0,1]$, the expected optimal assignment cost converges to $\zeta(2) = \pi^2/6$ (Mézard–Parisi via the cavity method, proved by Aldous, 2001) — combinatorial optimization solved by spin-glass techniques.
- **Scheduling and logistics.** Crew rostering, machine scheduling with deadlines, and airline fleet assignment are $b$-matchings and min-cost flows; total unimodularity is what makes them solvable at industrial scale.

### AI and machine learning

- **Optimal transport.** The assignment problem is discrete optimal transport with uniform marginals: minimize $\langle C, X \rangle$ over couplings with prescribed row and column sums. Adding entropy, $\min_X \langle C, X \rangle - \varepsilon H(X)$, gives the **Sinkhorn** algorithm — alternating row/column normalizations whose fixed point is $X = \mathrm{diag}(a) e^{-C/\varepsilon} \mathrm{diag}(b)$. As $\varepsilon \to 0$ the coupling converges to a permutation, recovering the Hungarian solution; at finite $\varepsilon$ it is smooth and differentiable, hence trainable end-to-end. Wasserstein distances, Sinkhorn divergences and WGAN critics all sit on this axis.
- **Attention as soft matching.** A softmax attention map is row-stochastic, not doubly stochastic: it lets many queries claim the same key. Sinkhorn attention and slot-attention style modules add the column normalization, moving the operator toward a genuine matching and imposing the "each key is used once" inductive bias that plain attention lacks.
- **Set prediction with Hungarian loss.** DETR-style detectors and set transformers score predictions against ground truth by first solving an $O(n^3)$ **assignment** between predicted and true objects, then back-propagating through the matched pairs. The matching itself is not differentiated; it fixes the permutation ambiguity of set-valued outputs.
- **Graph cuts for structured prediction.** Binary MRF/CRF energies with submodular pairwise terms are minimized **exactly** by a single min-cut computation (Boykov–Jolly, Kolmogorov–Zabih), giving MAP inference for image segmentation, stereo, and denoising; $\alpha$-expansion extends this to multi-label problems with approximation guarantees.
- **Fair and balanced routing.** Expert assignment in sparse mixture-of-experts layers is a capacitated assignment problem — auction/Sinkhorn-based balanced routing prevents expert collapse — and data-parallel batch balancing is a $b$-matching.
- **Graph alignment and embedding comparison.** Matching nodes across two graphs (network alignment, Gromov–Wasserstein) generalizes bipartite matching with structural costs; the quadratic version is NP-hard, so practitioners relax to Birkhoff and round — precisely because Birkhoff's vertices are permutations (Theorem I).

### A worked micro-example: flow, cut and matching in one picture

Network on $\lbrace s, a, b, t \rbrace$ with all capacities $1$: arcs $s \to a$, $s \to b$, $a \to b$, $a \to t$, $b \to t$.

1. **Greedy misstep.** Augment along $s \to a \to b \to t$: $\vert f \vert = 1$, and no path survives in the *original* graph.
2. **Residual repair.** $G_f$ now contains the reverse arc $b \to a$, giving the augmenting path $s \to b \to a \to t$ with bottleneck $1$. Push it: $f(a,b)$ drops back to $0$ and $\vert f \vert = 2$.
3. **Certificate.** In the new residual network, $s$ reaches nothing (both source arcs are saturated), so $S = \lbrace s \rbrace$, $T = \lbrace a,b,t \rbrace$ and

$$
c(S,T) = c(s,a) + c(s,b) = 2 = \vert f \vert
$$

4. **Matching reading.** Interpreting $\lbrace a, b \rbrace$ as workers on one side and $\lbrace a', b' \rbrace$ as jobs, the same computation says $\nu = 2$ and a minimum vertex cover has size $2$ — König in miniature.

$$
\boxed{\vert f^{\ast} \vert = 2 = c(\lbrace s \rbrace, \lbrace a,b,t \rbrace) = \nu = \tau}
$$

*The lesson in one line:* the reverse arc is what converts a stuck greedy solution into an optimal one, and the residual reachable set is what proves it.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Flow-value lemma, max-flow min-cut | Ford & Fulkerson (1956), *Canad. J. Math.* 8; CLRS (4th ed.) §24.2 |
| Residual networks and augmenting paths | Ahuja, Magnanti & Orlin, *Network Flows*, Ch. 6–7 |
| Edmonds–Karp $O(nm^2)$ analysis | Edmonds & Karp (1972), *J. ACM* 19(2); CLRS §24.3 |
| Dinic blocking flows, Hopcroft–Karp | Dinitz (1970); Hopcroft & Karp (1973), *SIAM J. Comput.* 2(4) |
| Push–relabel | Goldberg & Tarjan (1988), *J. ACM* 35(4) |
| Integrality and total unimodularity | Schrijver, *Combinatorial Optimization*, Ch. 5, 10 |
| Berge's augmenting-path theorem | Berge (1957), *PNAS* 43(9); Diestel §2.1 |
| König's theorem, minimum vertex cover | Kőnig (1931); West §3.1; Schrijver Ch. 16 |
| Hall's marriage theorem and deficiency | Hall (1935); Diestel §2.1; West §3.1 |
| Blossom algorithm for general matching | Edmonds (1965), *Canad. J. Math.* 17 |
| Birkhoff–von Neumann, assignment LP | Birkhoff (1946); Schrijver Ch. 18 |
| Hungarian algorithm and dual potentials | Kuhn (1955), *Naval Res. Logist. Q.* 2; Munkres (1957) |
| Menger's theorem via unit-capacity flow | Menger (1927); Diestel §3.3 |
| Entropic OT, Sinkhorn, Wasserstein losses | Cuturi (2013), NeurIPS; Peyré & Cuturi (2019), *FnTML* 11(5–6) |
| Graph cuts for MAP inference | Boykov & Jolly (2001), ICCV; Kolmogorov & Zabih (2004), *TPAMI* 26(2) |
| Random assignment and $\zeta(2)$ | Mézard & Parisi (1987); Aldous (2001), *Random Struct. Algorithms* 18(4) |
| Matching-based set prediction | Carion et al. (2020), *End-to-End Object Detection with Transformers*, ECCV |
| Graph representation learning context | Hamilton (2020), *Graph Representation Learning* |

**Cross-links within this repository**

- Shortest augmenting paths and BFS layering: [`../04_shortest_paths_algorithms/README.md`](../04_shortest_paths_algorithms/README.md)
- Exchange arguments and greedy optimality: [`../03_trees_and_minimum_spanning_trees/README.md`](../03_trees_and_minimum_spanning_trees/README.md)
- Connectivity prerequisites for Menger: [`../02_traversal_and_connectivity/README.md`](../02_traversal_and_connectivity/README.md)
- Runnable networkx flow and matching demos: [`../computation.ipynb`](../computation.ipynb)
- Solved problem set for this topic: [`exercises.ipynb`](exercises.ipynb)